# ESA TESTING

In [5]:
from pathlib import Path
from typing import Optional

import joblib
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize


# ============================================================
# CONFIG
# ============================================================

PROCESSED_DIR = Path("../datasets/processed/PAN2011_300")
ARTIFACT_DIR = Path("../artifacts/esa")

SOURCE_CHUNKS_PATH = PROCESSED_DIR / "source_chunks_lsa_esa.parquet"
SUSPICIOUS_CHUNKS_PATH = PROCESSED_DIR / "suspicious_chunks_lsa_esa.parquet"
SOURCE_CANONICAL_CHUNKS_PATH = PROCESSED_DIR / "source_chunks.parquet"

SUSPICIOUS_DOC_ID = "part14__suspicious-document06510.txt"

OUTPUT_CANDIDATES_PATH = PROCESSED_DIR / "esa_candidates_suspicious_doc.parquet"
OUTPUT_TOP_DOCS_MEAN_PATH = PROCESSED_DIR / "esa_top_source_documents_by_mean_score.parquet"
OUTPUT_TOP_DOCS_MAX_PATH = PROCESSED_DIR / "esa_top_source_documents_by_max_score.parquet"

# Change to True only when rebuilding the ESA source index.
BUILD_INDEX = True


# ============================================================
# LOAD CHUNKS
# ============================================================

def load_lsa_esa_chunks(
    path: Path,
    text_column: str = "lsa_esa_text",
) -> pd.DataFrame:
    df = pd.read_parquet(path)

    required_columns = {
        "chunk_id",
        "doc_id",
        "chunk_index",
        "start_char",
        "end_char",
        text_column,
    }

    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")

    df = df.copy()
    df[text_column] = df[text_column].fillna("").astype(str)
    df = df[df[text_column].str.strip() != ""].reset_index(drop=True)

    return df


# ============================================================
# BUILD ESA INDEX
# ============================================================

def build_esa_index(
    source_chunks_path: Path,
    artifact_dir: Path,
    text_column: str = "lsa_esa_text",
    max_features: int = 100_000,
    max_source_chunks: Optional[int] = None,
) -> None:
    """
    Build and save the collection-based ESA source index.

    In this implementation:
    - TF-IDF terms are the explicit semantic dimensions.
    - Source chunks are represented in that explicit concept space.
    - Query chunks are projected into the same space.
    - Cosine similarity is computed between suspicious chunks and source chunks.

    Creates:
    - esa_tfidf_vectorizer.joblib
    - source_esa_vectors.npz
    - source_esa_metadata.parquet
    - esa_config.joblib
    """

    artifact_dir = Path(artifact_dir)
    artifact_dir.mkdir(parents=True, exist_ok=True)

    source_df = load_lsa_esa_chunks(
        path=source_chunks_path,
        text_column=text_column,
    )

    if max_source_chunks is not None:
        source_df = source_df.head(max_source_chunks).reset_index(drop=True)

    source_texts = source_df[text_column].tolist()

    print(f"Loaded {len(source_df)} source chunks")
    print("Fitting ESA TF-IDF vectorizer...")

    vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.85,
        sublinear_tf=True,
        lowercase=True,
        strip_accents="unicode",
        norm="l2",
    )

    source_esa = vectorizer.fit_transform(source_texts)

    # Extra safety. TfidfVectorizer already normalizes with norm="l2",
    # but this keeps the cosine similarity assumption explicit.
    source_esa = normalize(source_esa, norm="l2", axis=1)

    print(f"ESA source matrix shape: {source_esa.shape}")

    metadata_columns = [
        "chunk_id",
        "doc_id",
        "chunk_index",
        "start_char",
        "end_char",
    ]

    optional_columns = ["file_name", "relative_path", "part", "word_count"]
    metadata_columns += [col for col in optional_columns if col in source_df.columns]

    source_metadata = source_df[metadata_columns].copy()

    print("Saving ESA artifacts...")

    joblib.dump(vectorizer, artifact_dir / "esa_tfidf_vectorizer.joblib")
    sparse.save_npz(artifact_dir / "source_esa_vectors.npz", source_esa)

    source_metadata.to_parquet(
        artifact_dir / "source_esa_metadata.parquet",
        index=False,
    )

    config = {
        "text_column": text_column,
        "max_features": max_features,
        "max_source_chunks": max_source_chunks,
        "source_chunks_path": str(source_chunks_path),
        "matrix_shape": source_esa.shape,
        "similarity": "cosine_similarity_via_l2_normalized_dot_product",
        "note": "Collection-based ESA-style retrieval over source chunks, not Wikipedia ESA.",
    }

    joblib.dump(config, artifact_dir / "esa_config.joblib")

    print(f"Saved ESA artifacts to: {artifact_dir}")


# ============================================================
# LOAD ESA INDEX
# ============================================================

def load_esa_index(artifact_dir: Path):
    artifact_dir = Path(artifact_dir)

    vectorizer_path = artifact_dir / "esa_tfidf_vectorizer.joblib"
    vectors_path = artifact_dir / "source_esa_vectors.npz"
    metadata_path = artifact_dir / "source_esa_metadata.parquet"
    config_path = artifact_dir / "esa_config.joblib"

    for path in [vectorizer_path, vectors_path, metadata_path, config_path]:
        if not path.exists():
            raise FileNotFoundError(f"Missing ESA artifact: {path}")

    print("Loading ESA artifacts...")

    vectorizer = joblib.load(vectorizer_path)
    source_esa = sparse.load_npz(vectors_path)
    source_metadata = pd.read_parquet(metadata_path)
    config = joblib.load(config_path)

    return vectorizer, source_esa, source_metadata, config


# ============================================================
# QUERY ONE SUSPICIOUS DOCUMENT
# ============================================================

def retrieve_esa_candidates_for_suspicious_doc(
    suspicious_chunks_path: Path,
    artifact_dir: Path,
    suspicious_doc_id: str,
    output_path: Path,
    text_column: str = "lsa_esa_text",
    top_k: int = 20,
    batch_size: int = 8,
    max_suspicious_chunks: Optional[int] = None,
) -> pd.DataFrame:
    """
    Query the ESA source index using one selected suspicious document.

    Returns top-k source chunks for each suspicious chunk.
    """

    vectorizer, source_esa, source_metadata, config = load_esa_index(artifact_dir)

    suspicious_df = load_lsa_esa_chunks(
        path=suspicious_chunks_path,
        text_column=text_column,
    )

    suspicious_df = suspicious_df[
        suspicious_df["doc_id"] == suspicious_doc_id
    ].copy()

    if suspicious_df.empty:
        raise ValueError(f"No suspicious chunks found for doc_id: {suspicious_doc_id}")

    if max_suspicious_chunks is not None:
        suspicious_df = suspicious_df.head(max_suspicious_chunks).reset_index(drop=True)

    print(f"Selected suspicious document: {suspicious_doc_id}")
    print(f"Suspicious chunks to query: {len(suspicious_df)}")
    print(f"Searching top-{top_k} source chunks per suspicious chunk")

    results = []

    for start in range(0, len(suspicious_df), batch_size):
        end = min(start + batch_size, len(suspicious_df))
        batch_df = suspicious_df.iloc[start:end]

        batch_texts = batch_df[text_column].tolist()

        suspicious_esa = vectorizer.transform(batch_texts)
        suspicious_esa = normalize(suspicious_esa, norm="l2", axis=1)

        # Sparse cosine similarity:
        # because both matrices are L2-normalized, dot product = cosine similarity.
        similarities = suspicious_esa @ source_esa.T

        for local_i, suspicious_row in enumerate(batch_df.itertuples(index=False)):
            sims_sparse = similarities.getrow(local_i)

            if sims_sparse.nnz == 0:
                continue

            candidate_indices = sims_sparse.indices
            candidate_scores = sims_sparse.data

            safe_top_k = min(top_k, len(candidate_scores))

            top_local_indices = np.argpartition(
                -candidate_scores,
                safe_top_k - 1,
            )[:safe_top_k]

            top_local_indices = top_local_indices[
                np.argsort(-candidate_scores[top_local_indices])
            ]

            for rank, local_idx in enumerate(top_local_indices, start=1):
                source_idx = int(candidate_indices[local_idx])
                score = float(candidate_scores[local_idx])

                source_row = source_metadata.iloc[source_idx]

                results.append({
                    "suspicious_chunk_id": suspicious_row.chunk_id,
                    "suspicious_doc_id": suspicious_row.doc_id,
                    "suspicious_chunk_index": suspicious_row.chunk_index,
                    "suspicious_start_char": suspicious_row.start_char,
                    "suspicious_end_char": suspicious_row.end_char,

                    "source_chunk_id": source_row["chunk_id"],
                    "source_doc_id": source_row["doc_id"],
                    "source_chunk_index": source_row["chunk_index"],
                    "source_start_char": source_row["start_char"],
                    "source_end_char": source_row["end_char"],

                    "ESA_score": score,
                    "ESA_rank": rank,
                })

        print(f"Processed suspicious chunks {start} to {end}")

    output_df = pd.DataFrame(results)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_df.to_parquet(output_path, index=False)

    print(f"Saved ESA candidates to: {output_path}")

    return output_df


# ============================================================
# DOCUMENT-LEVEL MEAN SCORE
# ============================================================

def get_top_source_documents_by_mean_esa_score(
    candidates_df: pd.DataFrame,
    source_chunks_path: Path,
    suspicious_doc_id: str,
    top_n: int = 20,
    min_match_count: int = 4,
    output_path: Optional[Path] = None,
) -> pd.DataFrame:
    """
    Rank unique source documents by mean ESA score.
    """

    required_columns = {
        "suspicious_doc_id",
        "suspicious_chunk_id",
        "source_doc_id",
        "source_chunk_id",
        "ESA_score",
    }

    missing = required_columns - set(candidates_df.columns)
    if missing:
        raise ValueError(f"Missing columns in candidates_df: {missing}")

    filtered_df = candidates_df[
        candidates_df["suspicious_doc_id"] == suspicious_doc_id
    ].copy()

    if filtered_df.empty:
        raise ValueError(f"No candidates found for suspicious_doc_id: {suspicious_doc_id}")

    grouped_df = (
        filtered_df
        .groupby("source_doc_id")
        .agg(
            mean_ESA_score=("ESA_score", "mean"),
            max_ESA_score=("ESA_score", "max"),
            min_ESA_score=("ESA_score", "min"),
            match_count=("ESA_score", "count"),
            unique_source_chunks=("source_chunk_id", "nunique"),
            unique_suspicious_chunks=("suspicious_chunk_id", "nunique"),
        )
        .reset_index()
    )

    grouped_df = grouped_df[grouped_df["match_count"] >= min_match_count].copy()

    grouped_df = (
        grouped_df
        .sort_values(
            ["mean_ESA_score", "match_count", "max_ESA_score"],
            ascending=[False, False, False],
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    grouped_df["source_doc_rank"] = range(1, len(grouped_df) + 1)

    grouped_df = grouped_df[
        [
            "source_doc_rank",
            "source_doc_id",
            "mean_ESA_score",
            "max_ESA_score",
            "min_ESA_score",
            "match_count",
            "unique_source_chunks",
            "unique_suspicious_chunks",
        ]
    ]

    source_meta_df = pd.read_parquet(
        source_chunks_path,
        columns=["doc_id", "relative_path"],
    ).drop_duplicates("doc_id")

    source_meta_df = source_meta_df.rename(columns={
        "doc_id": "source_doc_id",
        "relative_path": "source_relative_path",
    })

    grouped_df = grouped_df.merge(
        source_meta_df,
        on="source_doc_id",
        how="left",
    )

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        grouped_df.to_parquet(output_path, index=False)
        print(f"Saved top source documents to: {output_path}")

    return grouped_df


# ============================================================
# DOCUMENT-LEVEL MAX SCORE
# ============================================================

def get_top_source_documents_by_max_esa_score(
    candidates_df: pd.DataFrame,
    source_chunks_path: Path,
    suspicious_doc_id: str,
    top_n: int = 20,
    min_match_count: int = 1,
    output_path: Optional[Path] = None,
) -> pd.DataFrame:
    """
    Rank unique source documents by strongest ESA chunk match.

    This is usually better for plagiarism source retrieval because plagiarism
    is often local: one strong passage-level match can identify the true source.
    """

    required_columns = {
        "suspicious_doc_id",
        "suspicious_chunk_id",
        "source_doc_id",
        "source_chunk_id",
        "ESA_score",
    }

    missing = required_columns - set(candidates_df.columns)
    if missing:
        raise ValueError(f"Missing columns in candidates_df: {missing}")

    filtered_df = candidates_df[
        candidates_df["suspicious_doc_id"] == suspicious_doc_id
    ].copy()

    if filtered_df.empty:
        raise ValueError(f"No candidates found for suspicious_doc_id: {suspicious_doc_id}")

    grouped_df = (
        filtered_df
        .groupby("source_doc_id")
        .agg(
            mean_ESA_score=("ESA_score", "mean"),
            max_ESA_score=("ESA_score", "max"),
            min_ESA_score=("ESA_score", "min"),
            match_count=("ESA_score", "count"),
            unique_source_chunks=("source_chunk_id", "nunique"),
            unique_suspicious_chunks=("suspicious_chunk_id", "nunique"),
        )
        .reset_index()
    )

    grouped_df = grouped_df[grouped_df["match_count"] >= min_match_count].copy()

    grouped_df = (
        grouped_df
        .sort_values(
            ["max_ESA_score", "match_count", "mean_ESA_score"],
            ascending=[False, False, False],
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    grouped_df["source_doc_rank"] = range(1, len(grouped_df) + 1)

    grouped_df = grouped_df[
        [
            "source_doc_rank",
            "source_doc_id",
            "mean_ESA_score",
            "max_ESA_score",
            "min_ESA_score",
            "match_count",
            "unique_source_chunks",
            "unique_suspicious_chunks",
        ]
    ]

    source_meta_df = pd.read_parquet(
        source_chunks_path,
        columns=["doc_id", "relative_path"],
    ).drop_duplicates("doc_id")

    source_meta_df = source_meta_df.rename(columns={
        "doc_id": "source_doc_id",
        "relative_path": "source_relative_path",
    })

    grouped_df = grouped_df.merge(
        source_meta_df,
        on="source_doc_id",
        how="left",
    )

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        grouped_df.to_parquet(output_path, index=False)
        print(f"Saved top source documents to: {output_path}")

    return grouped_df


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    BUILD_INDEX = False
    SUSPICIOUS_DOC_ID = "part1__suspicious-document00007.txt"

    if BUILD_INDEX:
        build_esa_index(
            source_chunks_path=SOURCE_CHUNKS_PATH,
            artifact_dir=ARTIFACT_DIR,
            text_column="lsa_esa_text",
            max_features=100_000,
            max_source_chunks=None,
        )

    candidates_df = retrieve_esa_candidates_for_suspicious_doc(
        suspicious_chunks_path=SUSPICIOUS_CHUNKS_PATH,
        artifact_dir=ARTIFACT_DIR,
        suspicious_doc_id=SUSPICIOUS_DOC_ID,
        output_path=OUTPUT_CANDIDATES_PATH,
        text_column="lsa_esa_text",
        top_k=500,
        batch_size=8,
        max_suspicious_chunks=None,
    )

    top_sources_mean_df = get_top_source_documents_by_mean_esa_score(
        candidates_df=candidates_df,
        source_chunks_path=SOURCE_CANONICAL_CHUNKS_PATH,
        suspicious_doc_id=SUSPICIOUS_DOC_ID,
        top_n=50,
        min_match_count=1,
        output_path=OUTPUT_TOP_DOCS_MEAN_PATH,
    )

    top_sources_max_df = get_top_source_documents_by_max_esa_score(
        candidates_df=candidates_df,
        source_chunks_path=SOURCE_CANONICAL_CHUNKS_PATH,
        suspicious_doc_id=SUSPICIOUS_DOC_ID,
        top_n=50,
        min_match_count=1,
        output_path=OUTPUT_TOP_DOCS_MAX_PATH,
    )

    #print("\nTOP SOURCE DOCUMENTS BY MAX ESA SCORE")
    #print(top_sources_max_df.to_string(index=False))

    #print("\nTOP SOURCE DOCUMENTS BY MEAN ESA SCORE")
    #print(top_sources_mean_df.to_string(index=False))

top_sources_max_df

Loading ESA artifacts...
Selected suspicious document: part1__suspicious-document00007.txt
Suspicious chunks to query: 9
Searching top-500 source chunks per suspicious chunk
Processed suspicious chunks 0 to 8
Processed suspicious chunks 8 to 9
Saved ESA candidates to: ..\datasets\processed\PAN2011_300\esa_candidates_suspicious_doc.parquet
Saved top source documents to: ..\datasets\processed\PAN2011_300\esa_top_source_documents_by_mean_score.parquet
Saved top source documents to: ..\datasets\processed\PAN2011_300\esa_top_source_documents_by_max_score.parquet


,source_doc_rank,source_doc_id,mean_ESA_score,max_ESA_score,min_ESA_score,match_count,unique_source_chunks,unique_suspicious_chunks,source_relative_path
0,1,part13__source-document06022.txt,0.109079,0.726295,0.065011,395,249,9,part13/source-document06022.txt
1,2,part19__source-document09499.txt,0.086391,0.146896,0.067365,68,51,6,part19/source-document09499.txt
2,3,part18__source-document08528.txt,0.086963,0.128673,0.071686,81,66,4,part18/source-document08528.txt
3,4,part23__source-document11043.txt,0.080477,0.123862,0.065631,53,40,7,part23/source-document11043.txt
4,5,part10__source-document04702.txt,0.088332,0.123314,0.064915,51,29,5,part10/source-document04702.txt
5,6,part20__source-document09559.txt,0.104678,0.116460,0.092896,2,2,1,part20/source-document09559.txt
6,7,part9__source-document04194.txt,0.081075,0.115616,0.067204,55,45,8,part9/source-document04194.txt
7,8,part15__source-document07246.txt,0.088312,0.114613,0.076623,5,5,3,part15/source-document07246.txt
8,9,part8__source-document03838.txt,0.100363,0.113636,0.087089,2,2,1,part8/source-document03838.txt
9,10,part2__source-document00537.txt,0.084136,0.113627,0.066171,30,24,4,part2/source-document00537.txt
